# 10. Fine-tuning małego modelu Transformers

**Cel:** w pełni lokalnie wstępnie nauczyć mały BERT zadania samonadzorowanego, zapisać wagi, wczytać je jako bazę klasyfikatora i dostroić. Czas: 20–30 min. Wymaga `torch`, `transformers`.

**Uruchamianie:** wykonuj komórki kolejno. Dane są generowane lokalnie; nie jest potrzebny internet.

### Wagi bazowe z zadania maskowania
Budujemy model o losowej inicjalizacji (`BertForMaskedLM`), wstępnie uczymy przewidywać zamaskowany token, a następnie zapisujemy go lokalnie. To mały eksperyment dydaktyczny: wagi bazowe nie pochodzą z publicznego modelu językowego. Tokeny 1–8 to zabawkowy słownik; 9 to maska, 10 to CLS.

In [ ]:
import tempfile
from pathlib import Path
import torch
from transformers import BertConfig,BertForMaskedLM,BertForSequenceClassification
torch.manual_seed(12);torch.set_num_threads(1)
config=BertConfig(vocab_size=16,hidden_size=32,num_hidden_layers=1,num_attention_heads=2,
                  intermediate_size=64,max_position_embeddings=16,type_vocab_size=2,
                  hidden_dropout_prob=0,attention_probs_dropout_prob=0,pad_token_id=0)
base=BertForMaskedLM(config)
optimizer=torch.optim.AdamW(base.parameters(),lr=0.003)
gen=torch.Generator().manual_seed(13)
pretrain_losses=[]
for step in range(40):
    raw=torch.randint(1,9,(32,4),generator=gen)
    # w sekwencji [CLS,a,b,a,b] zamaskuj powtórzony token a na pozycji 3
    seq=torch.stack([torch.full((32,),10),raw[:,0],raw[:,1],raw[:,0],raw[:,1]],dim=1)
    target=torch.full_like(seq,-100);target[:,3]=seq[:,3]
    seq[:,3]=9
    base.train();optimizer.zero_grad()
    loss=base(input_ids=seq,labels=target).loss
    loss.backward();optimizer.step()
    pretrain_losses.append(float(loss))
print('Strata maskowania:',round(pretrain_losses[0],3),'->',round(pretrain_losses[-1],3))
assert pretrain_losses[-1]<pretrain_losses[0]


### Dostrojenie klasyfikatora
Wczytujemy warstwy bazowe z plików lokalnych. Nowa głowa klasyfikacyjna jest inicjalizowana od zera (Transformers może wyświetlić oczekiwane ostrzeżenie). Sprawdzamy zmianę wag enkodera, a nie tylko spadek straty głowy.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    base.save_pretrained(tmp)
    classifier=BertForSequenceClassification.from_pretrained(tmp,num_labels=2,ignore_mismatched_sizes=True)
    before=classifier.bert.embeddings.word_embeddings.weight.detach().clone()
    train_raw=torch.randint(1,9,(160,4),generator=gen)
    train_labels=(train_raw[:,0]>4).long()
    train_ids=torch.cat([torch.full((160,1),10),train_raw],dim=1)
    test_raw=torch.randint(1,9,(80,4),generator=torch.Generator().manual_seed(14))
    test_labels=(test_raw[:,0]>4).long()
    test_ids=torch.cat([torch.full((80,1),10),test_raw],dim=1)
    opt=torch.optim.AdamW(classifier.parameters(),lr=0.003)
    losses=[]
    for epoch in range(50):
        classifier.train();opt.zero_grad()
        result=classifier(input_ids=train_ids,labels=train_labels)
        result.loss.backward();opt.step()
        losses.append(float(result.loss))
    classifier.eval()
    with torch.no_grad():
        accuracy=float((classifier(input_ids=test_ids).logits.argmax(-1)==test_labels).float().mean())
    change=float((classifier.bert.embeddings.word_embeddings.weight-before).abs().max())
    print('Strata klasyfikacji:',round(losses[0],3),'->',round(losses[-1],3))
    print('Test accuracy:',round(accuracy,3),'największa zmiana wagi embeddingu:',round(change,6))
    assert losses[-1]<losses[0]
    assert accuracy>0.80
    assert change>0


**Analiza:** Zablokuj bazę `classifier.bert.requires_grad_(False)` przed treningiem i porównaj liczbę zmienionych wag oraz accuracy. Czy model trenowany na sztucznych tokenach pozwala wyciągać wnioski o języku naturalnym? Nie: to ćwiczenie mechaniki dostrajania.